In [3]:
"""
============================================================
WEEK 2: Outlier / Anomaly Detection
------------------------------------------------------------
Goal: Flag days where Revenue deviated significantly from the
expected norm using 3 complementary techniques:
  1. Z-Score analysis
  2. Interquartile Range (IQR)
  3. Isolation Forest

Each flagged day is then contextualized (weekend? public holiday?
demand spike/drop?) so the numbers turn into a real explanation,
not just a list of dates.
============================================================
"""

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
from sklearn.ensemble import IsolationForest

In [4]:
sns.set_theme(style="whitegrid")

In [5]:
# 1. Load all 3 files
files = [
    "Customers_A_ID12346-14312.xlsx",
    "Customers_B_ID14314-16283.xlsx",
    "Customers_C_ID16284-18287.xlsx",
]
dfs = [pd.read_excel(f) for f in files]


In [6]:
# 2. Combine into one DataFrame
df = pd.concat(dfs, ignore_index=True)
print("Combined shape:", df.shape)
df.head(50)

Combined shape: (406829, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,550644,22722,SET OF 6 SPICE TINS PANTRY DESIGN,7,2011-04-19 16:19:00,3.95,12733,USA
1,550644,22979,PANTRY WASHING UP BRUSH,2,2011-04-19 16:19:00,1.45,12733,USA
2,550644,84987,SET OF 36 TEATIME PAPER DOILIES,3,2011-04-19 16:19:00,1.45,12733,USA
3,550644,22720,SET OF 3 CAKE TINS PANTRY DESIGN,1,2011-04-19 16:19:00,4.95,12733,USA
4,550644,22993,SET OF 4 PANTRY JELLY MOULDS,1,2011-04-19 16:19:00,1.25,12733,USA
5,550644,47580,TEA TIME DES TEA COSY,3,2011-04-19 16:19:00,2.55,12733,USA
6,550644,22989,SET 2 PANTRY DESIGN TEA TOWELS,6,2011-04-19 16:19:00,3.25,12733,USA
7,550644,22900,SET 2 TEA TOWELS I LOVE LONDON,3,2011-04-19 16:19:00,3.25,12733,USA
8,550644,22128,PARTY CONES CANDY ASSORTED,12,2011-04-19 16:19:00,1.25,12733,USA
9,550644,47590B,PINK HAPPY BIRTHDAY BUNTING,4,2011-04-19 16:19:00,5.45,12733,USA


In [9]:
# 1. Build a daily time series from the cleaned data
# Group all transactions by date and sum the Revenue for each day
daily_sales = df.groupby(df["InvoiceDate"].dt.date)["Revenue"].sum()

# Convert the index to a proper datetime type
daily_sales.index = pd.to_datetime(daily_sales.index)

daily_sales = daily_sales.sort_index()
daily_sales.head()

InvoiceDate
2010-12-01    13717.33
2010-12-02    45795.77
2010-12-03    22620.46
2010-12-04       20.34
2010-12-05    31400.94
Name: Revenue, dtype: float64

In [11]:
# ------------------------------------------------------------------
# 1. Z-SCORE METHOD
# ------------------------------------------------------------------
# Measures how many standard deviations each day's revenue is from
# the mean. |z| > 3 is a common rule-of-thumb threshold for outliers.

z_scores = stats.zscore(daily_sales)
z_threshold = 3

z_outliers = daily_sales[np.abs(z_scores) > z_threshold]

print(f"Z-Score method flagged {len(z_outliers)} anomalous day(s) (|z| > {z_threshold}):")
z_outliers

Z-Score method flagged 39 anomalous day(s) (|z| > 3):


InvoiceDate
2010-12-02     45795.77
2010-12-07     53151.49
2010-12-16     48026.38
2011-01-11     59475.32
2011-05-12     57928.14
2011-05-17     45518.21
2011-07-19     45814.39
2011-07-28     53594.11
2011-08-04     57771.05
2011-08-11     54095.25
2011-08-17     46526.36
2011-08-18     53261.55
2011-09-13     47979.53
2011-09-15     69696.44
2011-09-19     46223.84
2011-09-20    103400.56
2011-09-22     55619.90
2011-10-03     60139.11
2011-10-05     73779.03
2011-10-06     52822.99
2011-10-07     48893.36
2011-10-20     60735.75
2011-10-21     60153.13
2011-10-27     46033.07
2011-11-03     59944.47
2011-11-04     53206.59
2011-11-09     57962.14
2011-11-10     68473.07
2011-11-14     56825.88
2011-11-15     47517.59
2011-11-16     46893.24
2011-11-17     55114.84
2011-11-22     49093.73
2011-11-23     70362.35
2011-11-28     50116.15
2011-11-29     48476.60
2011-12-05     56637.83
2011-12-07     68992.92
2011-12-08     49454.94
Name: Revenue, dtype: float64

In [12]:
# ------------------------------------------------------------------
# 2. IQR (INTERQUARTILE RANGE) METHOD
# ------------------------------------------------------------------
# More robust to skewed distributions than Z-score since it uses
# quartiles instead of mean/std. Standard rule: anything beyond
# 1.5 * IQR from Q1/Q3 is an outlier.

Q1 = daily_sales.quantile(0.25)
Q3 = daily_sales.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

iqr_outliers = daily_sales[(daily_sales < lower_bound) | (daily_sales > upper_bound)]

print(f"IQR method flagged {len(iqr_outliers)} anomalous day(s)")
print(f"Normal range: {lower_bound:,.2f} to {upper_bound:,.2f}")
iqr_outliers

IQR method flagged 320 anomalous day(s)
Normal range: -78.30 to 142.50


InvoiceDate
2010-12-01    13717.33
2010-12-02    45795.77
2010-12-03    22620.46
2010-12-05    31400.94
2010-12-06    30480.38
                ...   
2012-11-29      244.08
2012-11-30      144.00
2013-04-19     1627.20
2013-04-29      175.20
2014-04-12      204.00
Name: Revenue, Length: 320, dtype: float64